# Deep Learning 005 — The Perceptron Trick

The update rule is one line: **when you get a point wrong, move the line toward it.**
We watch it work on clean data, then watch it never settle on real data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def step(z):
    return (z >= 0).astype(int)

def train_perceptron(X, y, lr=0.1, epochs=20, seed=0, record_every=None):
    rng = np.random.default_rng(seed)
    w = rng.normal(size=X.shape[1]) * 0.01
    b = 0.0
    history, snaps = [], []
    for ep in range(epochs):
        for i in rng.permutation(len(X)):
            pred = step(X[i] @ w + b)
            err = y[i] - pred            # +1, 0 or -1
            w = w + lr * err * X[i]      # the trick
            b = b + lr * err
        history.append((step(X @ w + b) == y).mean())
        if record_every and ep % record_every == 0:
            snaps.append((ep, w.copy(), b))
    return w, b, np.array(history), snaps

`err` is `+1` when the point should have fired and did not, `-1` in the other
direction, and `0` when the prediction was right. So a correct point produces **no
update at all** — the line only moves in response to mistakes.

## On separable data it converges, and then stops moving

In [ ]:
rng = np.random.default_rng(1)
A = rng.normal([2, 2], 0.6, size=(60, 2))
B = rng.normal([-1, -1], 0.6, size=(60, 2))
Xs = np.vstack([A, B])
ys = np.r_[np.ones(60, int), np.zeros(60, int)]

w, b, hist, snaps = train_perceptron(Xs, ys, lr=0.1, epochs=30,
                                     record_every=3)
print('accuracy per epoch:', np.round(hist, 3))
print(f'\nfinal accuracy {hist[-1]:.1%}')
print(f'epochs after which nothing changed: '
      f'{int((hist == hist[-1]).sum())} of {len(hist)}')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.9))
ax[0].scatter(Xs[ys == 1, 0], Xs[ys == 1, 1], marker='^', s=18)
ax[0].scatter(Xs[ys == 0, 0], Xs[ys == 0, 1], marker='o', s=18)
xs = np.linspace(Xs[:, 0].min() - .5, Xs[:, 0].max() + .5, 50)
for ep, ww, bb in snaps:
    ax[0].plot(xs, -(ww[0] / ww[1]) * xs - bb / ww[1], lw=1,
               alpha=0.5, label=f'epoch {ep}')
ax[0].plot(xs, -(w[0] / w[1]) * xs - b / w[1], 'k-', lw=2, label='final')
ax[0].set(title='the line moving, separable data',
          ylim=(Xs[:, 1].min() - .5, Xs[:, 1].max() + .5))
ax[0].legend(fontsize=7)
ax[1].plot(hist, 'o-'); ax[1].set(xlabel='epoch', ylabel='accuracy',
                                  title='converges and stays', ylim=(0.4, 1.05))
plt.tight_layout(); plt.show()

## On real data it never settles

`placement.csv` is *nearly* separable but not quite. The perceptron has no notion of
"good enough" — every remaining mistake produces another update, forever.

In [ ]:
df = pd.read_csv('../data/placement.csv')
X = df[['cgpa', 'resume_score']].to_numpy()
y = df['placed'].to_numpy().astype(int)

w2, b2, hist2, _ = train_perceptron(X, y, lr=0.1, epochs=60, seed=0)
print(f'best epoch accuracy   {hist2.max():.1%}  (epoch {int(hist2.argmax())})')
print(f'final epoch accuracy  {hist2[-1]:.1%}')
print(f'std dev over last 30 epochs: {hist2[-30:].std():.4f}')
print(f'distinct accuracies in the last 30 epochs: {len(set(np.round(hist2[-30:], 4)))}')

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.plot(hist2, lw=1.2)
ax.axhline(hist2.max(), color='seagreen', ls='--',
           label=f'best seen {hist2.max():.1%}')
ax.set(xlabel='epoch', ylabel='accuracy',
       title='placement.csv: it oscillates and never converges')
ax.legend(); plt.tight_layout(); plt.show()

**This is the perceptron's defining limitation, and it is not about accuracy.** It is
that the algorithm has no objective it is minimising — only a rule that fires on
mistakes. With no separating line to find, it wanders. Nothing tells it that the line
at epoch 41 was better than the one at epoch 42.

That missing ingredient is a **loss function**, which is lesson 006.

## Exercises

1. Lower `lr` to 0.001 and re-run on `placement.csv`. Does the oscillation shrink? Does
   it stop?
2. Add a rule that keeps the best weights seen so far (a "pocket" perceptron). What
   accuracy do you get?
3. Make the separable data barely separable by moving the two clusters closer. How many
   epochs to converge? Plot epochs-to-converge against cluster separation.